# FQL Succession Gate B — E-uni Sanity (Colab) [rev.1 notebook]

> 文档对应：
> - Spec：[`docs/fql_succession_p0p1_spec.md`](../docs/fql_succession_p0p1_spec.md) §5 (Task E — Gate B FQL E-uni sanity)
> - Plan：[`docs/fql_succession_plan_v0.md`](../docs/fql_succession_plan_v0.md) §3 (Lean MVP)
> - Dataset card：[`docs/fql_e_uni_anchor_dataset_card.md`](../docs/fql_e_uni_anchor_dataset_card.md)
> - Pre-flight Task A：[`docs/fql_audit_dryrun_report.md`](../docs/fql_audit_dryrun_report.md) (audit tool verified, Gate A.2 PASS on 200-ep dry-run)
> - Pre-flight Gate A.1：[`docs/rebrac_broad_validation_v2_report.md`](../docs/rebrac_broad_validation_v2_report.md) §2 (arrival_v2 reward bridge HOLDS at 0.85)
>
> **本 notebook rev.1 (2026-05-19)** — 风格 mirror `rebrac_broad_validation_v2_core.ipynb` rev.3：
> - `os.system` 让 train / eval 输出 line-buffer stream
> - skip-resume：train 看 `trainer_state.json + agent_final.pt`，eval 看 `results/.../test_result.json`
> - eval 数字落 `results/offline/fql_succession/gate_b/<run>/test_result.json`（不在 checkpoint dir）
> - 汇总 → `results/offline/fql_succession/gate_b/summaries/{gate_b_verdict.json, gate_b_overview.csv}`
>
> 目的：在 E-uni 1000-ep paper anchor 上跑 **paired** ReBRAC + FQL × seed=42 sanity，应用 spec §5.3 四条 Gate B 判据决定是否进入 P2 main comparison。

## 执行摘要（2 paired runs）

| Run | Algo | Dataset | Manifest | Steps | Seed | Notes |
|---|---|---|---|---|---|---|
| 1 | **ReBRAC** | `privileged_s0_h4_arrival_v2_re150_u10cross_fixdone_ep1000` | `single_u10_cross_tgt15` | 200k | 42 | paired baseline，与 broad val v2 N0 同 cell 但本 notebook 用 privileged 数据集（FQL paper anchor 即此） |
| 2 | **FQL** | 同上 | 同上 | 200k | 42 | flow-matching teacher + 1-step student，主对照 |

**Wallclock 预算**：200k step × 2 run ≈ **2.5–3h L4**（FQL flow-matching 比 ReBRAC 慢 1.3–1.5×；in-training eval 20 × 100 ep ≈ 30 min 额外/run）。

## 输出位置

| 类型 | 路径 | Sync 回 git? |
|---|---|---|
| Train artifacts | `checkpoints/offline/fql_succession/gate_b/<algo>_e_uni_seed42/{agent_final.pt, trainer_state.json, train_log.jsonl, eval_log.csv}` | 否（大文件，gitignore） |
| Final eval | `results/offline/fql_succession/gate_b/<algo>_e_uni_seed42/test_result.json` | **是** |
| In-training eval traj | `results/offline/fql_succession/gate_b/<algo>_e_uni_seed42/eval_log.csv`（从 checkpoint dir copy） | **是** |
| 汇总 | `results/offline/fql_succession/gate_b/summaries/{gate_b_verdict.json, gate_b_overview.csv}` | **是** |

## Gate B 判据 (spec §5.3, pre-registered)

| # | 指标 | 通过条件 |
|---|---|---|
| c1 | FQL `test_success` last 3 eval mean | **≥ ReBRAC `test_success` last 3 eval mean − 0.08** |
| c2 | FQL `loss_flow` 末段（last 5% train log mean） | **< 0.05**（teacher 收敛） |
| c3 | FQL `actor_loss` 末段数量级 | **vs ReBRAC `actor_loss` 末段在 ±1 OOM 内** |
| c4 | FQL eval 曲线 monotonicity | **末 30% 训练 success rate 线性 trend ≥ 0**（no collapse） |

**4 条全过 → Gate B pass → 进入 P2 main comparison spec 写作。**

任一失败的处理 (spec §5.3)：
- FQL << ReBRAC：实现 bug，debug 3 iter；仍失败 → STOP (plan v1 R4 mitigation)
- `loss_flow` 不收敛：检查 time sampling / flow path / teacher lr
- 曲线 collapse：检查 Q-normalization / actor lr / `distill_alpha_bc`

## Spec CLI ↔ 实际代码 字段映射（rev.1 不动 spec）

本 notebook 用**实际代码**字段名；spec 写作时的字段名已 deprecated 但 spec 待 Task E 闭环后统一更新到 v1.1。

| Spec 名称 | 实际名称 |
|---|---|
| `--algorithm` | `--algo` |
| `--eval-every-steps` | `--eval-every` |
| `--rebrac-actor-bc-coef` | `--actor-penalty-coef` |
| `--rebrac-critic-bc-coef` | `--critic-penalty-coef` |
| `--rebrac-critic-use-layernorm` | `--critic-layernorm` |
| `--fql-flow-steps` | `--flow-steps` |
| `--fql-distill-alpha-bc` | `--distill-alpha-bc` |

Dataset path：spec 的 placeholder `offline_data/fql_succession/e_uni_1000/` → 实际采用现有 naming convention `offline_data/privileged_s0_h4_arrival_v2_re150_u10cross_fixdone_ep1000/`。

## 0. 环境 sanity

In [ ]:
!lscpu | head -8
print()
!nvidia-smi

In [ ]:
import torch
print(f'PyTorch       : {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device 0      : {torch.cuda.get_device_name(0)}')
    print(f'cuDNN         : {torch.backends.cudnn.version()}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
%cd /content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5

In [ ]:
!pwd
!ls scripts/train_offline.py scripts/evaluate_offline.py
!ls docs/fql_succession_p0p1_spec.md docs/fql_e_uni_anchor_dataset_card.md
!ls auv_nav/fql.py auv_nav/rebrac.py
!ls offline_data/privileged_s0_h4_arrival_v2_re150_u10cross_fixdone_ep1000/transitions.npz \
    offline_data/privileged_s0_h4_arrival_v2_re150_u10cross_fixdone_ep1000/metadata.json
!ls benchmarks/single_u10_cross_tgt15.json
!ls wake_data/wake_v8_U1p00_Re150_D12p00_dx0p60_Ti5pct_1200f_roi.npy \
    wake_data/wake_v8_U1p00_Re150_D12p00_dx0p60_Ti5pct_1200f_roi_meta.json

In [ ]:
# Optional sanity: confirm FQL unit tests still pass on the Colab runtime
# (skip with shift+enter if you've already verified locally).
!python -m pytest tests/test_fql.py -q --no-header 2>&1 | tail -20

## 1. Run matrix

2 runs = ReBRAC + FQL × seed=42。两个 run 共用：
- 同 dataset（E-uni 1000-ep privileged）
- 同 eval manifest（`single_u10_cross_tgt15`，100 ep）
- 同 total-steps（200k）
- 同 eval-every（10k step，20 个 eval point）
- 同 sampling-mode（`uniform`，spec §5.2 没用 `shuffle_no_replacement`；与 broad val v2 不同）

差异只在 algo + algo-specific CLI flags：
- ReBRAC：`--actor-penalty-coef 4.0 --critic-penalty-coef 2.0 --critic-layernorm --no-actor-layernorm`
- FQL：`--flow-steps 10 --distill-alpha-bc 1.0`（teacher_lr / time_embed_dim 用默认 3e-4 / 32）

Skip-resume：
- **train 阶段** 看 `<ckpt_dir>/trainer_state.json` + `<ckpt_dir>/agent_final.pt` 都存在则跳过
- **eval 阶段** 看 `<result_dir>/test_result.json` 存在则跳过

In [ ]:
import os
from pathlib import Path

REPO_ROOT = Path('.').resolve()
CKPT_ROOT = REPO_ROOT / 'checkpoints' / 'offline' / 'fql_succession' / 'gate_b'
RESULT_ROOT = REPO_ROOT / 'results' / 'offline' / 'fql_succession' / 'gate_b'
SUMMARIES_DIR = RESULT_ROOT / 'summaries'
SUMMARIES_DIR.mkdir(parents=True, exist_ok=True)

DATASET_DIR = 'offline_data/privileged_s0_h4_arrival_v2_re150_u10cross_fixdone_ep1000'
DATASET_NPZ = f'{DATASET_DIR}/transitions.npz'
FLOW_PATH = 'wake_data/wake_v8_U1p00_Re150_D12p00_dx0p60_Ti5pct_1200f_roi.npy'
MANIFEST = 'benchmarks/single_u10_cross_tgt15.json'
SEED = 42
TOTAL_STEPS = 200_000
EVAL_EVERY = 10_000
EVAL_EPISODES = 100

ALGOS = ['rebrac', 'fql']

RUNS = []
for algo in ALGOS:
    run_id = f'{algo}_e_uni_seed{SEED}'
    ckpt_dir = CKPT_ROOT / run_id
    result_dir = RESULT_ROOT / run_id
    RUNS.append({
        'run_id': run_id,
        'algo': algo,
        'seed': SEED,
        'dataset': DATASET_NPZ,
        'manifest': MANIFEST,
        'flow': FLOW_PATH,
        'ckpt_dir': str(ckpt_dir),
        'result_dir': str(result_dir),
    })

print(f'{"#":>2} {"algo":<8} {"seed":>5}  ckpt_dir / result_dir')
print('-' * 110)
for i, r in enumerate(RUNS, 1):
    print(f'{i:>2} {r["algo"]:<8} {r["seed"]:>5}  ckpt:   {r["ckpt_dir"]}')
    print(f'   {"":<8} {"":>5}  result: {r["result_dir"]}')

## 2. Training (2 runs, ~2.5–3h L4 total)

**关键差异 vs broad val v2 notebook**：本 notebook 训练时**保留** `--eval-every 10000 --eval-episodes 100`，因为 Gate B c1 / c4 需要 in-training eval 轨迹（last 3 eval mean + last 30% monotonicity）。

训练输出文件：
- `<ckpt_dir>/train_log.jsonl`：每 `--log-every`（默认 1000 step）的 loss metrics（含 FQL `loss_flow`）
- `<ckpt_dir>/eval_log.csv`：每 `--eval-every` 的 in-training eval（100 ep）→ Gate B c1/c4 数据源
- `<ckpt_dir>/agent_final.pt`：最终 checkpoint
- `<ckpt_dir>/trainer_state.json`：完整 trainer state（含 algo / agent_config，用于 evaluate_offline 自动 dispatch）

Per-algo CLI 构造：`_common_cli()` 拼共有 flag，再按 algo 加 algo-specific flag。

In [ ]:
import shlex
import time

def _common_cli(r):
    return [
        'python', '-u', '-m', 'scripts.train_offline',
        '--algo', r['algo'],
        '--offline-data', r['dataset'],
        '--flow', r['flow'],
        '--manifest', r['manifest'],
        '--probe-layout', 's0',
        '--history-length', '4',
        '--task-geometry', 'cross_stream',
        '--target-speed', '1.5',
        '--objective', 'arrival_v2',
        '--total-steps', str(TOTAL_STEPS),
        '--batch-size', '256',
        '--sampling-mode', 'uniform',
        '--hidden-dim', '256',
        '--num-hidden-layers', '3',  # _resolve_num_hidden_layers returns 3 for both rebrac and fql
        '--actor-lr', '3e-4',
        '--critic-lr', '3e-4',
        '--gamma', '0.99',
        '--tau', '0.005',
        '--policy-noise', '0.2',
        '--noise-clip', '0.5',
        '--policy-freq', '2',
        '--grad-clip-norm', '10.0',
        '--normalizer-eps', '1e-3',
        '--eval-every', str(EVAL_EVERY),
        '--eval-episodes', str(EVAL_EPISODES),
        '--eval-workers', '4',
        '--eval-worker-device', 'cpu',
        '--log-every', '1000',
        '--checkpoint-every', '0',  # only emit agent_final.pt; skip intermediate ckpts
        '--seed', str(r['seed']),
        '--device', 'cuda',
        '--save-dir', r['ckpt_dir'],
    ]

def _algo_flags(algo):
    if algo == 'rebrac':
        return [
            '--actor-penalty-coef', '4.0',
            '--critic-penalty-coef', '2.0',
            '--critic-layernorm',
            '--no-actor-layernorm',
        ]
    if algo == 'fql':
        return [
            '--flow-steps', '10',
            '--distill-alpha-bc', '1.0',
            '--teacher-lr', '3e-4',
            '--flow-time-embed-dim', '32',
        ]
    raise ValueError(f'Unknown algo: {algo}')

for i, r in enumerate(RUNS, 1):
    ckpt_dir = Path(r['ckpt_dir'])
    trainer_state = ckpt_dir / 'trainer_state.json'
    agent_final = ckpt_dir / 'agent_final.pt'

    print(f'\n========== [{i}/{len(RUNS)}] train {r["algo"]} seed={r["seed"]} ==========')

    if trainer_state.exists() and agent_final.exists():
        print(f'[skip] already complete: {ckpt_dir}')
        continue

    ckpt_dir.mkdir(parents=True, exist_ok=True)
    t0 = time.time()

    cmd_parts = _common_cli(r) + _algo_flags(r['algo'])
    cmd_str = ' '.join(shlex.quote(p) for p in cmd_parts)
    print(cmd_str)
    ret = os.system(cmd_str)  # os.system → line-buffered stream (matches v2_core rev.2 fix)
    elapsed = (time.time() - t0) / 60
    if ret != 0:
        print(f'\n[FAIL] {r["algo"]} seed={r["seed"]} exit={ret} ({elapsed:.1f} min)')
        break
    print(f'[done] {r["algo"]} seed={r["seed"]} → {ckpt_dir} ({elapsed:.1f} min)')

## 3. Final evaluation (2 runs × 100 episodes against fixed manifest)

训练内 eval 已经写在 `eval_log.csv`（最后一行 = train_step=200000 的 eval），这里 separately 再跑一次 `scripts.evaluate_offline` 作为 **canonical final eval**（与 broad val v2 protocol 一致）。

`evaluate_offline` 通过 `trainer_state.json` 自动 dispatch 到正确的 agent class（rebrac / fql），无需手动指定 algo。

In [ ]:
for i, r in enumerate(RUNS, 1):
    ckpt_dir = Path(r['ckpt_dir'])
    result_dir = Path(r['result_dir'])
    test_json = result_dir / 'test_result.json'

    print(f'\n========== [{i}/{len(RUNS)}] eval {r["algo"]} seed={r["seed"]} ==========')

    if test_json.exists():
        print(f'[skip] test_result.json exists: {test_json}')
        continue
    if not (ckpt_dir / 'agent_final.pt').exists():
        print(f'[skip] no agent_final.pt at {ckpt_dir} (training not done?)')
        continue

    result_dir.mkdir(parents=True, exist_ok=True)

    cmd_parts = [
        'python', '-u', '-m', 'scripts.evaluate_offline',
        '--checkpoint', str(ckpt_dir),
        '--agent-file', 'agent_final.pt',
        '--manifest', r['manifest'],
        '--episodes', '100',
        '--seed', '123',
        '--device', 'cuda',
        '--num-workers', '4',
        '--worker-device', 'cpu',
        '--output-json', str(test_json),
    ]
    cmd_str = ' '.join(shlex.quote(p) for p in cmd_parts)
    print(cmd_str)
    ret = os.system(cmd_str)
    if ret != 0:
        print(f'[FAIL] {r["algo"]} seed={r["seed"]} eval exit={ret}')
        break
    print(f'[done] {test_json}')

## 3.5 Archive in-training eval_log.csv → results/

Gate B c1 (last 3 eval mean) + c4 (last 30% monotonicity) 都依赖 `eval_log.csv`。`eval_log.csv` 由训练写入 checkpoint dir，但 checkpoint dir 是 gitignored —— 把它 copy 到 `results/` 一份，sync 回 git 留痕。

In [ ]:
import shutil
for r in RUNS:
    src = Path(r['ckpt_dir']) / 'eval_log.csv'
    dst = Path(r['result_dir']) / 'eval_log.csv'
    if not src.exists():
        print(f'[noop] {src} missing (training not done)')
        continue
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    print(f'[archive] {src} → {dst}')

## 4. Raw diagnostic — test_result.json 字段 + eval_log.csv 末 3 行 + train_log.jsonl 末 5 行

在跑 §5 verdict 前打印 raw 数字，便于排查 verdict 出 bug 时定位是哪一步出问题。

In [ ]:
import csv
import json

for r in RUNS:
    print(f'\n=== {r["algo"]} seed={r["seed"]} ===')

    # 4a. test_result.json (final 100-ep eval)
    test_json = Path(r['result_dir']) / 'test_result.json'
    print(f'\n[a] test_result.json @ {test_json}')
    if not test_json.exists():
        print('  MISSING')
    else:
        d = json.loads(test_json.read_text())
        for k in sorted(d.keys()):
            v = d[k]
            if isinstance(v, (int, float, str, bool, type(None))):
                print(f'  {k}: {v}')
            elif isinstance(v, list):
                print(f'  {k}: list(len={len(v)})')
            elif isinstance(v, dict):
                print(f'  {k}: dict(keys={list(v.keys())[:5]}{"..." if len(v) > 5 else ""})')

    # 4b. eval_log.csv 末 3 行
    eval_csv = Path(r['result_dir']) / 'eval_log.csv'
    if not eval_csv.exists():
        eval_csv = Path(r['ckpt_dir']) / 'eval_log.csv'
    print(f'\n[b] eval_log.csv (last 3) @ {eval_csv}')
    if not eval_csv.exists():
        print('  MISSING')
    else:
        with eval_csv.open() as fp:
            rows = list(csv.DictReader(fp))
        for row in rows[-3:]:
            step = row.get('train_step', '?')
            succ = row.get('eval_success_rate', '?')
            ret_ = row.get('eval_return', '?')
            prog = row.get('eval_progress_ratio', '?')
            print(f'  step={step}  success={succ}  return={ret_}  progress={prog}')

    # 4c. train_log.jsonl 末 5 行（FQL 看 loss_flow，ReBRAC 看 loss_actor）
    train_jsonl = Path(r['ckpt_dir']) / 'train_log.jsonl'
    print(f'\n[c] train_log.jsonl (last 5) @ {train_jsonl}')
    if not train_jsonl.exists():
        print('  MISSING')
    else:
        with train_jsonl.open() as fp:
            lines = fp.readlines()
        for line in lines[-5:]:
            row = json.loads(line)
            step = row.get('train_step', '?')
            keys_of_interest = ['critic_loss', 'actor_loss', 'bc_loss', 'mean_q', 'lambda',
                                 'loss_flow', 'td_abs_error', 'critic_penalty']
            vals = ' '.join(
                f'{k}={row[k]:.4f}' if isinstance(row.get(k), (int, float)) else ''
                for k in keys_of_interest if k in row
            )
            print(f'  step={step}  {vals.strip()}')

## 5. Gate B verdict — 4 criteria

判据来源 spec §5.3（pre-registered）：

| # | 指标 | 数据源 | 通过条件 |
|---|---|---|---|
| c1 | FQL last-3-eval success mean | `eval_log.csv` 末 3 行的 `eval_success_rate` mean | ≥ ReBRAC same − 0.08 |
| c2 | FQL loss_flow 末段 | `train_log.jsonl` 末 5% 行的 `loss_flow` mean | < 0.05 |
| c3 | FQL actor_loss 末段 OOM | `train_log.jsonl` 末 5% 行的 `actor_loss` mean | vs ReBRAC same，比值在 [0.1, 10] |
| c4 | FQL eval 曲线 monotonicity | `eval_log.csv` 末 30% 行的 `eval_success_rate` 线性 trend | slope ≥ 0 |

**4 条全过 → Gate B PASS → 进入 P2 main comparison spec**。任一失败 → 按 spec §5.3 mitigation 处理。

In [ ]:
import csv
import json
import math
import statistics

def _load_eval_log(run):
    """Return list of dicts from eval_log.csv (prefer results/, fall back to ckpt/)."""
    for candidate in [Path(run['result_dir']) / 'eval_log.csv',
                       Path(run['ckpt_dir']) / 'eval_log.csv']:
        if candidate.exists():
            with candidate.open() as fp:
                return list(csv.DictReader(fp))
    return []

def _load_train_jsonl(run):
    path = Path(run['ckpt_dir']) / 'train_log.jsonl'
    if not path.exists():
        return []
    with path.open() as fp:
        return [json.loads(line) for line in fp if line.strip()]

def _safe_float(x):
    try:
        f = float(x)
        return None if math.isnan(f) else f
    except (TypeError, ValueError):
        return None

def _linear_slope(ys):
    n = len(ys)
    if n < 2:
        return 0.0
    xs = list(range(n))
    mean_x = sum(xs) / n
    mean_y = sum(ys) / n
    num = sum((x - mean_x) * (y - mean_y) for x, y in zip(xs, ys))
    den = sum((x - mean_x) ** 2 for x in xs)
    return num / den if den > 0 else 0.0

# ---- Extract Gate B inputs ----
summary = {'runs': {}}
for r in RUNS:
    eval_rows = _load_eval_log(r)
    train_rows = _load_train_jsonl(r)

    succ_traj = [_safe_float(row.get('eval_success_rate')) for row in eval_rows]
    succ_traj = [s for s in succ_traj if s is not None]
    last3_mean = statistics.mean(succ_traj[-3:]) if len(succ_traj) >= 3 else (
        statistics.mean(succ_traj) if succ_traj else float('nan'))
    last30pct_idx = max(1, int(0.7 * len(succ_traj)))
    last30pct_slope = _linear_slope(succ_traj[last30pct_idx:]) if succ_traj else 0.0

    # 末段 = 末 5% rows of train_log.jsonl
    tail_n = max(1, int(0.05 * len(train_rows)))
    tail = train_rows[-tail_n:]
    loss_flow_tail = [_safe_float(row.get('loss_flow')) for row in tail]
    loss_flow_tail = [v for v in loss_flow_tail if v is not None]
    actor_loss_tail = [_safe_float(row.get('actor_loss')) for row in tail]
    actor_loss_tail = [v for v in actor_loss_tail if v is not None]

    summary['runs'][r['algo']] = {
        'seed': r['seed'],
        'n_eval_points': len(succ_traj),
        'eval_success_trajectory': succ_traj,
        'last3_eval_success_mean': last3_mean,
        'last30pct_eval_slope': last30pct_slope,
        'n_train_log_rows': len(train_rows),
        'loss_flow_last_5pct_mean': statistics.mean(loss_flow_tail) if loss_flow_tail else float('nan'),
        'actor_loss_last_5pct_mean': statistics.mean(actor_loss_tail) if actor_loss_tail else float('nan'),
    }

# ---- Pretty-print extracted Gate B inputs ----
print('=' * 100)
print('Gate B raw inputs:')
for algo, s in summary['runs'].items():
    print(f'\n  {algo} (seed={s["seed"]}):')
    print(f'    n_eval_points          = {s["n_eval_points"]}')
    print(f'    last3_eval_success_mean = {s["last3_eval_success_mean"]:.4f}')
    print(f'    last30pct_eval_slope    = {s["last30pct_eval_slope"]:+.5f}')
    print(f'    n_train_log_rows       = {s["n_train_log_rows"]}')
    print(f'    loss_flow_last_5pct    = {s["loss_flow_last_5pct_mean"]:.4f}')
    print(f'    actor_loss_last_5pct   = {s["actor_loss_last_5pct_mean"]:+.4f}')

# ---- Apply Gate B criteria ----
rebrac = summary['runs'].get('rebrac', {})
fql = summary['runs'].get('fql', {})

rebrac_last3 = rebrac.get('last3_eval_success_mean', float('nan'))
fql_last3 = fql.get('last3_eval_success_mean', float('nan'))
fql_loss_flow = fql.get('loss_flow_last_5pct_mean', float('nan'))
fql_actor_loss = fql.get('actor_loss_last_5pct_mean', float('nan'))
rebrac_actor_loss = rebrac.get('actor_loss_last_5pct_mean', float('nan'))
fql_slope = fql.get('last30pct_eval_slope', 0.0)

# c1: FQL last-3 ≥ ReBRAC last-3 − 0.08
c1_pass = (fql_last3 >= rebrac_last3 - 0.08) if not (math.isnan(fql_last3) or math.isnan(rebrac_last3)) else False
c1_gap = fql_last3 - rebrac_last3

# c2: FQL loss_flow < 0.05
c2_pass = (fql_loss_flow < 0.05) if not math.isnan(fql_loss_flow) else False

# c3: FQL/ReBRAC actor_loss within ±1 OOM
if not math.isnan(fql_actor_loss) and not math.isnan(rebrac_actor_loss) and abs(rebrac_actor_loss) > 1e-9:
    ratio = abs(fql_actor_loss) / abs(rebrac_actor_loss)
    c3_pass = (0.1 <= ratio <= 10.0)
else:
    ratio = float('nan')
    c3_pass = False

# c4: FQL last 30% eval slope ≥ 0
c4_pass = fql_slope >= 0

verdict_block = {
    'c1_last3_parity': {
        'fql_last3': fql_last3, 'rebrac_last3': rebrac_last3,
        'gap_fql_minus_rebrac': c1_gap,
        'threshold': -0.08, 'pass': bool(c1_pass),
    },
    'c2_loss_flow_converged': {
        'fql_loss_flow_last_5pct_mean': fql_loss_flow,
        'threshold': 0.05, 'pass': bool(c2_pass),
    },
    'c3_actor_loss_oom_parity': {
        'fql_actor_loss': fql_actor_loss,
        'rebrac_actor_loss': rebrac_actor_loss,
        'abs_ratio_fql_over_rebrac': ratio,
        'threshold': '[0.1, 10.0]', 'pass': bool(c3_pass),
    },
    'c4_eval_monotonicity_last_30pct': {
        'fql_slope': fql_slope, 'threshold': 0.0, 'pass': bool(c4_pass),
    },
    'overall_pass': bool(c1_pass and c2_pass and c3_pass and c4_pass),
}
summary['verdict'] = verdict_block

print()
print('=' * 100)
print('Gate B verdict:')
print(f'  c1 last-3 parity:        FQL={fql_last3:.4f} vs ReBRAC={rebrac_last3:.4f} '
      f'(Δ={c1_gap:+.4f}, threshold ≥ −0.08) → {"PASS" if c1_pass else "FAIL"}')
print(f'  c2 loss_flow converged:  FQL loss_flow={fql_loss_flow:.4f} '
      f'(threshold < 0.05) → {"PASS" if c2_pass else "FAIL"}')
print(f'  c3 actor_loss OOM:       |FQL|/|ReBRAC|={ratio:.3f} '
      f'(threshold ∈ [0.1, 10]) → {"PASS" if c3_pass else "FAIL"}')
print(f'  c4 monotonicity:         FQL last-30% slope={fql_slope:+.5f} '
      f'(threshold ≥ 0) → {"PASS" if c4_pass else "FAIL"}')
print()
print(f'  OVERALL: {"Gate B PASS" if verdict_block["overall_pass"] else "Gate B FAIL"}')

# ---- Write verdict + overview to results/ ----
verdict_path = SUMMARIES_DIR / 'gate_b_verdict.json'
verdict_path.write_text(json.dumps(summary, indent=2, default=str))
print(f'\n[wrote] {verdict_path}')

csv_path = SUMMARIES_DIR / 'gate_b_overview.csv'
with csv_path.open('w', newline='') as fp:
    w = csv.writer(fp)
    w.writerow(['algo', 'seed', 'last3_eval_success_mean', 'last30pct_eval_slope',
                 'loss_flow_last_5pct_mean', 'actor_loss_last_5pct_mean', 'n_eval_points'])
    for algo, s in summary['runs'].items():
        w.writerow([algo, s['seed'],
                     f'{s["last3_eval_success_mean"]:.4f}',
                     f'{s["last30pct_eval_slope"]:+.5f}',
                     f'{s["loss_flow_last_5pct_mean"]:.4f}',
                     f'{s["actor_loss_last_5pct_mean"]:+.4f}',
                     s['n_eval_points']])
print(f'[wrote] {csv_path}')

## 5.5 Final 100-ep test_result.json side-by-side (供 paper anchor 引用)

Gate B 判据 c1 用的是 in-training last-3 eval mean（spec §5.3 强调避免 single-shot fluctuation）。但 paper 写作时通常 cite **single canonical final-eval number**——把 §3 写入的 `test_result.json` 也列出来便于参考。

In [ ]:
import json
print('=' * 100)
print(f'{"algo":<8} {"success":>10} {"return":>10} {"safety":>10} {"prog":>10} {"path_eff":>10}')
print('-' * 100)
for r in RUNS:
    test_json = Path(r['result_dir']) / 'test_result.json'
    if not test_json.exists():
        print(f'{r["algo"]:<8} MISSING')
        continue
    d = json.loads(test_json.read_text())
    succ = d.get('eval_success_rate', d.get('success_rate', float('nan')))
    ret_ = d.get('eval_avg_return', d.get('avg_return', float('nan')))
    safety = d.get('eval_avg_safety_cost', d.get('avg_safety_cost', float('nan')))
    prog = d.get('eval_avg_progress_ratio', d.get('avg_progress_ratio', float('nan')))
    path_eff = d.get('eval_avg_path_efficiency', d.get('avg_path_efficiency', float('nan')))
    print(f'{r["algo"]:<8} {succ:>10.4f} {ret_:>10.2f} {safety:>10.3f} {prog:>10.4f} {path_eff:>10.4f}')
print('=' * 100)
print("\nReBRAC anchor reference (broad val v2 N0, crosscomp dataset, 2 seed): 0.850 ± 0.024")
print('FQL Gate B target on this dataset: ≥ ReBRAC last-3 − 0.08')

## 6. 跑完后清单（回到 local）

### Sync 回 git 仓库（只取 `results/`）

```bash
# 在本地 repo root：
rsync -av '<drive>/results/offline/fql_succession/gate_b/' \
  results/offline/fql_succession/gate_b/
```

包含：
- `results/offline/fql_succession/gate_b/{rebrac,fql}_e_uni_seed42/test_result.json` × 2
- `results/offline/fql_succession/gate_b/{rebrac,fql}_e_uni_seed42/eval_log.csv` × 2
- `results/offline/fql_succession/gate_b/summaries/{gate_b_verdict.json, gate_b_overview.csv}`

**checkpoint 不动**（agent_final.pt 大文件，gitignore；只在 P2 spec 想 reuse seed=42 anchor 时回到 Drive）。

### 后续步骤

1. **写** `docs/fql_succession_gate_b_report.md`：4 criteria 数字 + verdict + raw_inputs + paired comparison + paper-anchor 数字
2. **Verdict-conditional 分支**：
   - **Gate B PASS**：升级 `docs/fql_succession_plan_v0.md` 到 v1.1，开始 P2 main comparison spec（M-multi-mix dataset 收集 + FQL/ReBRAC 主对照）
   - **任一 criterion FAIL**：按 spec §5.3 mitigation——
     - c1 FAIL（FQL << ReBRAC）：debug FQL implementation（3 iter），仍失败 STOP
     - c2 FAIL（loss_flow 不收敛）：teacher_lr / flow_steps / time embedding 排查
     - c3 FAIL（actor_loss OOM 偏离）：Q-normalization / lambda_coef / distill_alpha_bc 排查
     - c4 FAIL（曲线 collapse）：actor_lr 调小或 warm-up，重跑 1 seed
3. **Spec CLI rename**（deferred）：Task E 闭环后 patch `docs/fql_succession_p0p1_spec.md` §5.1/§5.2 的 CLI 字段名对齐实际代码（--algorithm→--algo / --eval-every-steps→--eval-every / --fql-*→--* / --rebrac-*-bc-coef→--actor/critic-penalty-coef / --rebrac-critic-use-layernorm→--critic-layernorm）
4. **N0 1000-ep audit（可选 paper appendix evidence）**：Gate B 顺利通过后，用 §Task A audit 工具在 N0 cell 跑 1000-ep audit（同 cell 不同 episode count）作为 paper §appendix evidence